# 编译 pass Hack：构建、加载、过滤与回滚

本 Notebook 复查已保存的 CPU 原生实验；不在当前内核重建或替换 jaxlib。完整 native/wheel 身份审计已在固定构建镜像执行，命令与记录见 `pass-event-build-002/roundtrip-validation-*`。TPU 设备事件仍需独立实验。

In [1]:
from pathlib import Path
import hashlib, json, sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research/jax-stack/status.json").is_file())
sys.path.insert(0, str(root / "research/jax-stack"))
from matmul_probe import fingerprint
from verify_extensions import audit
from pass_event_checks import analyze
from pass_events_probe import read_events
from verify_pass_hack import audit_test_xml
import numpy as np
import xml.etree.ElementTree as ET
result = json.loads((root / "research/jax-stack/pass-hack-results.json").read_text())
assert result["build_load_rollback_accepted"]
for name, expected in result["artifacts"].items():
    assert fingerprint(root / name) == expected
assert fingerprint(root / result["patch"]["path"]) == {k: result["patch"][k] for k in ["sha256", "size_bytes"]}
print("已保存的验证记录、补丁及原始证据指纹一致；本内核执行归档复查。")

已保存的验证记录、补丁及原始证据指纹一致；本内核执行归档复查。


## 1. 修改范围与 C++ 控制

只在 enabled leaf pass 的 `RunHelper` 周围插入 TraceMe。保留原 StatusOr 返回值；pipeline 包装层、filter、外部 dump 和 invariant 检查不计入该区间。

In [2]:
raw = root / "artifacts/jax-stack/pass-event-build-002"
cpp = audit_test_xml(ET.parse(raw / "unit-test.xml").getroot())
print("C++ tests:", cpp["tests"], "failures:", cpp["failures"])
print("\n".join(cpp["new_tests"]))
rollback = json.loads((raw / "rollback.json").read_text())
assert rollback["source_restored"] and rollback["git_status_after"] == ""
for path, expected in rollback["restored_files"].items():
    assert fingerprint(raw / "restored" / path) == expected
print("三个源文件已恢复为原始字节。")

C++ tests: 25 failures: 0
ResearchDisabledRecorderPreservesPassResult
ResearchErrorEventStopsAndPreservesStatus
ResearchFilteredLeafDoesNotEmitEvent
ResearchLeafEventsDescribeChangesAndNestedPipeline
三个源文件已恢复为原始字节。


## 2. 同一输入的三状态原始 trace

Generic algsimp 在 filter 前创建，因此禁用后仍有 3 个外层事件。真正 RunHelper 的自定义 algsimp 在过滤后为 0。

In [3]:
for state, modes in result["states"].items():
    for mode, saved in modes.items():
        capture = root / saved["capture"]
        assert fingerprint(capture / "manifest.json") == saved["manifest"]
        checked = audit(capture)
        expected = "present" if state == "patched" else "absent"
        observed = analyze(read_events(capture), saved["observations"]["compiled_module"], expected,
                           "algsimp" if mode == "filtered" else None)
        assert observed == saved["observations"]
        algsimp = sum(g["count"] for g in observed["groups"] if g["pass"] == "algsimp")
        print(state, mode, "custom:", observed["custom_event_count"],
              "custom algsimp:", algsimp, "generic algsimp:", observed["generic_known_pass_counts"]["algsimp"],
              "artifacts:", checked["artifact_count"])

baseline default custom: 0 custom algsimp: 0 generic algsimp: 3 artifacts: 16
baseline filtered custom: 0 custom algsimp: 0 generic algsimp: 3 artifacts: 16
patched default custom: 127 custom algsimp: 3 generic algsimp: 3 artifacts: 16
patched filtered custom: 124 custom algsimp: 0 generic algsimp: 3 artifacts: 16
rollback default custom: 0 custom algsimp: 0 generic algsimp: 3 artifacts: 16
rollback filtered custom: 0 custom algsimp: 0 generic algsimp: 3 artifacts: 16


## 3. 独立数值参考

输入为 float32 A[64,128] 与 W[128,64]，参考在 NumPy float64 中计算 tanh(A @ W)。三次 warm 输出均逐项复验。

In [4]:
for state, modes in result["states"].items():
    for mode, saved in modes.items():
        capture = root / saved["capture"]
        with np.load(capture / "inputs.npz", allow_pickle=False) as values:
            reference = np.tanh(values["a"].astype(np.float64) @ values["w"].astype(np.float64))
        with np.load(capture / "outputs.npz", allow_pickle=False) as values:
            errors = []
            for i in range(3):
                actual = values[f"output_{i}"]
                np.testing.assert_allclose(actual, reference, rtol=2e-5, atol=2e-5)
                errors.append(float(np.max(np.abs(actual - reference))))
        assert errors == saved["max_absolute_errors"]
        print(state, mode, "max absolute error:", max(errors))

baseline default max absolute error: 5.551717559629243e-09
baseline filtered max absolute error: 5.551717559629243e-09
patched default max absolute error: 5.551717559629243e-09
patched filtered max absolute error: 5.551717559629243e-09
rollback default max absolute error: 5.551717559629243e-09
rollback filtered max absolute error: 5.551717559629243e-09


## 4. XSpace 与导出 JSON 的信息差异

`program_id` 是内部 stat，转换器跳过它。最终补丁保留原字段，同时写入 `research_program_id`。以下使用真实失败 capture 验证门槛，不能用“已经有事件”替代完整身份。

In [5]:
loss = json.loads((raw / "program-id-loss.json").read_text())
print({k: loss[k] for k in ["xspace_custom_event_count", "xspace_program_id_present_count", "exported_program_id_present_count"]})
failed = root / "artifacts/jax-stack/pass-patch-runtime-001/patched-env/default"
try:
    analyze(read_events(failed), "jit_pass_event_workload", "present")
except ValueError as error:
    assert "metadata missing" in str(error)
    print("真实失败记录仍被拒绝:", error)
else:
    raise AssertionError("缺少导出身份的旧记录被错误接受")

{'xspace_custom_event_count': 127, 'xspace_program_id_present_count': 127, 'exported_program_id_present_count': 0}
真实失败记录仍被拒绝: custom event metadata missing


## 重新进行 native 身份审计

在 [源码构建说明](source-build.md) 指定的固定镜像、只读源码和基础 venv 挂载下运行：

```bash
.venv/bin/python -B research/jax-stack/verify_pass_hack.py --selftest
```

新建实验使用 [prepare_pass_runtime.py](prepare_pass_runtime.py)，传入成功的 build manifest、`--expected-events present` 及新输出目录。详见 [补丁说明](pass-event-patch.md) 和 [验收规则](pass-event-acceptance.md)。